In [ ]:
# ============================================================
# Clean install
# ============================================================
!pip uninstall -y autogluon autogluon.timeseries chronos-forecasting gluonts torch torchvision torchaudio -q

!pip install -U \
  "torch>=2.6,<2.10" \
  "autogluon.timeseries>=1.5,<1.6" \
  "chronos-forecasting>=2.0" -q

In [ ]:
# ============================================================
# IMPORTANT
# ============================================================
print("Now restart the runtime/session manually, then run Cell 3 onward.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 3: Verify imports and versions
# ============================================================
import torch
import pandas as pd
import numpy as np

from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("AutoGluon TimeSeries imported successfully.")

In [ ]:
# ============================================================
# CELL 2: Imports + settings + load data + prepare splits
# ============================================================
import os
import math
import warnings
import numpy as np
import pandas as pd

from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

warnings.filterwarnings("ignore")

# -----------------------------
# Paths
# -----------------------------
CSV_PATH = #Complete path of the csv dataset file goes here
SAVE_DIR = #Path to store the Output file goes here
os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------
# Main columns
# -----------------------------
DATE_COL = "WeekStartDate"
TARGET_COL = "WeeklyCrimeCount"
ID_COL = "DistricId"
NAME_COL = "DistrictName"

# -----------------------------
# Time split
# -----------------------------
TRAIN_START = "2019-01-01"
TRAIN_END   = "2022-12-31"

VAL_START   = "2023-01-01"
VAL_END     = "2023-12-31"

TEST_START  = "2024-01-01"
TEST_END    = "2024-12-31"

# -----------------------------
# Forecast settings
# -----------------------------
PREDICTION_LENGTH = 4
QUANTILES = [0.1, 0.5, 0.9]

# -----------------------------
# Chronos-2 settings
# -----------------------------
MODEL_PATH = "autogluon/chronos-2"
RANDOM_SEED = 42

# Start with safe fine-tuning values
FINE_TUNE_STEPS = 80
FINE_TUNE_LR = 1e-5
FINE_TUNE_BATCH_SIZE = 2
FINE_TUNE_CONTEXT_LENGTH = 256

# -----------------------------
# Outputs
# -----------------------------
VAL_LEADERBOARD_CSV = f"{SAVE_DIR}/chronos2_validation_leaderboard.csv"
TEST_METRICS_CSV = f"{SAVE_DIR}/chronos2_test_metrics.csv"
STEP_ERRORS_CSV = f"{SAVE_DIR}/chronos2_test_step_errors.csv"
PREDICTIONS_CSV = f"{SAVE_DIR}/chronos2_test_predictions.csv"

# -----------------------------
# Features
# -----------------------------
# future-known covariates
KNOWN_COVARIATES = [
    "WeekNo",
    "IsFirstWeek",
    "IsLastWeek",
    "unemployment_rate",
    "Week_inflation_rate",
    "isIslamicHoliday",
    "MonthNo",
    "YearofWeek",
]

# extra historical features / lag features
PAST_EXTRA_COLS = [
    "DistrictCrimeCountInMonth",
    "DistrictCrimeDensity",
    "Week_lag1",
    "Week_lag2",
    "Week_lag3",
    "Week_lag4",
    "week_rolling4",
]

ALL_FEATURE_COLS = KNOWN_COVARIATES + PAST_EXTRA_COLS

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(CSV_PATH)

required_cols = [DATE_COL, TARGET_COL, ID_COL, NAME_COL] + ALL_FEATURE_COLS
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL]).copy()

df[ID_COL] = df[ID_COL].astype(str)
df[NAME_COL] = df[NAME_COL].astype(str)

numeric_cols = [TARGET_COL] + ALL_FEATURE_COLS
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# target missing -> 0
df[TARGET_COL] = df[TARGET_COL].fillna(0.0)

# sort
df = df.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)

# fill feature missing values district-wise
df[ALL_FEATURE_COLS] = (
    df.groupby(ID_COL)[ALL_FEATURE_COLS]
      .transform(lambda x: x.ffill().bfill())
      .fillna(0.0)
)

# rename for AutoGluon
ts_df = df.rename(columns={
    ID_COL: "item_id",
    DATE_COL: "timestamp",
    TARGET_COL: "target"
}).copy()

keep_cols = ["item_id", "timestamp", "target"] + ALL_FEATURE_COLS
ts_df = ts_df[keep_cols].copy()

# split
train_df = ts_df[(ts_df["timestamp"] >= TRAIN_START) & (ts_df["timestamp"] <= TRAIN_END)].copy()
val_df   = ts_df[(ts_df["timestamp"] >= VAL_START) & (ts_df["timestamp"] <= VAL_END)].copy()
test_df  = ts_df[(ts_df["timestamp"] >= TEST_START) & (ts_df["timestamp"] <= TEST_END)].copy()

if train_df.empty:
    raise ValueError("Training data is empty.")
if val_df.empty:
    raise ValueError("Validation data is empty.")
if test_df.empty:
    raise ValueError("Test data is empty.")

# keep only districts common to all splits
common_ids = sorted(set(train_df["item_id"]) & set(val_df["item_id"]) & set(test_df["item_id"]))
train_df = train_df[train_df["item_id"].isin(common_ids)].copy()
val_df   = val_df[val_df["item_id"].isin(common_ids)].copy()
test_df  = test_df[test_df["item_id"].isin(common_ids)].copy()

if len(common_ids) == 0:
    raise ValueError("No common districts found across train, validation, and test.")

train_data = TimeSeriesDataFrame.from_data_frame(train_df, id_column="item_id", timestamp_column="timestamp")
val_data   = TimeSeriesDataFrame.from_data_frame(val_df,   id_column="item_id", timestamp_column="timestamp")
test_data  = TimeSeriesDataFrame.from_data_frame(test_df,  id_column="item_id", timestamp_column="timestamp")

district_name_map = (
    df[[ID_COL, NAME_COL]]
    .drop_duplicates()
    .assign(**{ID_COL: lambda x: x[ID_COL].astype(str)})
    .set_index(ID_COL)[NAME_COL]
    .to_dict()
)

print("Prepared successfully.")
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))
print("Districts:", len(common_ids))

In [ ]:
def compute_metrics(y_true, y_pred, q10=None, q90=None):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = math.sqrt(np.mean((y_true - y_pred) ** 2))

    nz = y_true != 0
    mape = np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100 if nz.any() else np.nan

    coverage = np.nan
    if q10 is not None and q90 is not None:
        q10 = np.asarray(q10, dtype=float)
        q90 = np.asarray(q90, dtype=float)
        coverage = np.mean((y_true >= q10) & (y_true <= q90)) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "Coverage": coverage,
        "N": len(y_true)
    }


In [ ]:
# ============================================================
# ARIMA validation-based order selection
# ============================================================
!pip install -q statsmodels

import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

# -----------------------------
# ARIMA outputs
# -----------------------------
ARIMA_SAVE_DIR = os.path.join(SAVE_DIR, "arima_results")
os.makedirs(ARIMA_SAVE_DIR, exist_ok=True)

ARIMA_VAL_RESULTS_CSV = f"{ARIMA_SAVE_DIR}/arima_validation_order_search.csv"
ARIMA_TEST_METRICS_CSV = f"{ARIMA_SAVE_DIR}/arima_test_metrics.csv"
ARIMA_STEP_ERRORS_CSV = f"{ARIMA_SAVE_DIR}/arima_test_step_errors.csv"
ARIMA_PREDICTIONS_CSV = f"{ARIMA_SAVE_DIR}/arima_test_predictions.csv"
ARIMA_BEST_ORDERS_CSV = f"{ARIMA_SAVE_DIR}/arima_best_orders.csv"

# -----------------------------
# Exogenous variables for ARIMA
# -----------------------------
ARIMA_EXOG_COLS = KNOWN_COVARIATES.copy()

# -----------------------------
# Candidate ARIMA orders
# Keep this modest so runtime stays manageable
# -----------------------------
ARIMA_ORDER_GRID = [
    (1, 0, 0),
    (0, 1, 1),
    (1, 1, 0),
    (1, 1, 1),
    (2, 1, 0),
    (0, 1, 2),
    (2, 1, 1),
    (2, 0, 2),
]

print("ARIMA exogenous columns:", ARIMA_EXOG_COLS)
print("Candidate orders:", ARIMA_ORDER_GRID)

def safe_fit_arima(y_train, exog_train, order):
    """
    Fit ARIMA safely and return fitted model or None.
    """
    try:
        model = ARIMA(
            endog=y_train,
            exog=exog_train,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fitted = model.fit()
        return fitted
    except Exception:
        return None

def rolling_validate_arima(hist_df, val_df_item, order, exog_cols, prediction_length=4):
    """
    Rolling 4-step validation on 2023.
    History starts with train (2016-2022), then expands with actual validation blocks.
    Returns MAE on validation horizon.
    """
    rolling_hist = hist_df.copy().sort_values("timestamp").reset_index(drop=True)
    val_item = val_df_item.copy().sort_values("timestamp").reset_index(drop=True)

    preds = []
    actuals = []

    start_idx = 0
    while start_idx < len(val_item):
        end_idx = min(start_idx + prediction_length, len(val_item))
        block_val = val_item.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        y_train = rolling_hist["target"].astype(float).values
        exog_train = rolling_hist[exog_cols].astype(float).values
        exog_future = block_val[exog_cols].astype(float).values

        fitted = safe_fit_arima(y_train, exog_train, order)
        if fitted is None:
            return np.inf

        try:
            pred = fitted.forecast(steps=len(block_val), exog=exog_future)
            pred = np.asarray(pred, dtype=float)
        except Exception:
            return np.inf

        preds.extend(pred.tolist())
        actuals.extend(block_val["target"].astype(float).tolist())

        # Expand history with ACTUAL validation block
        rolling_hist = pd.concat([rolling_hist, block_val], ignore_index=True)
        rolling_hist = rolling_hist.sort_values("timestamp").reset_index(drop=True)

        start_idx = end_idx

    preds = np.asarray(preds, dtype=float)
    actuals = np.asarray(actuals, dtype=float)
    mae = np.mean(np.abs(actuals - preds))
    return mae

# -----------------------------
# Validation-based order selection per district
# -----------------------------
arima_validation_rows = []
best_order_rows = []

for item_id in common_ids:
    train_item = train_df[train_df["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)
    val_item = val_df[val_df["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)

    if train_item.empty or val_item.empty:
        continue

    district_name = district_name_map.get(str(item_id), str(item_id))
    best_order = None
    best_mae = np.inf

    for order in ARIMA_ORDER_GRID:
        val_mae = rolling_validate_arima(
            hist_df=train_item,
            val_df_item=val_item,
            order=order,
            exog_cols=ARIMA_EXOG_COLS,
            prediction_length=PREDICTION_LENGTH
        )

        arima_validation_rows.append({
            "DistrictId": item_id,
            "DistrictName": district_name,
            "order": str(order),
            "Validation_MAE": val_mae
        })

        if np.isfinite(val_mae) and val_mae < best_mae:
            best_mae = val_mae
            best_order = order

    best_order_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "BestOrder": str(best_order) if best_order is not None else None,
        "BestValidationMAE": best_mae
    })

arima_val_results_df = pd.DataFrame(arima_validation_rows)
arima_best_orders_df = pd.DataFrame(best_order_rows)

arima_val_results_df.to_csv(ARIMA_VAL_RESULTS_CSV, index=False)
arima_best_orders_df.to_csv(ARIMA_BEST_ORDERS_CSV, index=False)

print("\nSaved validation order search:", ARIMA_VAL_RESULTS_CSV)
print("Saved best orders:", ARIMA_BEST_ORDERS_CSV)
display(arima_best_orders_df.head(20))

In [ ]:
# ============================================================
# ARIMA rolling 4-step forecast on 2024
# ============================================================
def compute_prediction_intervals(fitted_model, steps, exog_future, alpha=0.20):
    """
    Return mean forecast and 80% interval (alpha=0.20 => 10% / 90% style interval).
    """
    try:
        fc = fitted_model.get_forecast(steps=steps, exog=exog_future)
        mean_forecast = np.asarray(fc.predicted_mean, dtype=float)
        conf_int = fc.conf_int(alpha=alpha)

        # Handle both ndarray and DataFrame return types
        if isinstance(conf_int, pd.DataFrame):
            lower = conf_int.iloc[:, 0].astype(float).values
            upper = conf_int.iloc[:, 1].astype(float).values
        else:
            lower = np.asarray(conf_int[:, 0], dtype=float)
            upper = np.asarray(conf_int[:, 1], dtype=float)

        return mean_forecast, lower, upper
    except Exception:
        # fallback: forecast without intervals
        mean_forecast = np.asarray(fitted_model.forecast(steps=steps, exog=exog_future), dtype=float)
        lower = np.full(shape=steps, fill_value=np.nan, dtype=float)
        upper = np.full(shape=steps, fill_value=np.nan, dtype=float)
        return mean_forecast, lower, upper

# -----------------------------
# Build lookup for best orders
# -----------------------------
best_order_lookup = {}
for _, row in arima_best_orders_df.iterrows():
    item_id = str(row["DistrictId"])
    order_str = row["BestOrder"]
    if pd.notna(order_str) and order_str != "None":
        best_order_lookup[item_id] = eval(order_str)

print("Districts with selected ARIMA order:", len(best_order_lookup))

# -----------------------------
# History = train + validation
# Test = 2024
# -----------------------------
history_df_arima = pd.concat([train_df, val_df], ignore_index=True).copy()
history_df_arima = history_df_arima.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

test_df_sorted = test_df.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

all_arima_predictions = []
all_arima_errors = []
arima_district_rows = []

for item_id in common_ids:
    if item_id not in best_order_lookup:
        continue

    order = best_order_lookup[item_id]
    district_name = district_name_map.get(str(item_id), str(item_id))

    hist_item = history_df_arima[history_df_arima["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)
    test_item = test_df_sorted[test_df_sorted["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)

    if hist_item.empty or test_item.empty:
        continue

    rolling_hist = hist_item.copy()
    district_pred_parts = []

    start_idx = 0
    block_num = 1

    while start_idx < len(test_item):
        end_idx = min(start_idx + PREDICTION_LENGTH, len(test_item))
        block_test = test_item.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        y_train = rolling_hist["target"].astype(float).values
        exog_train = rolling_hist[ARIMA_EXOG_COLS].astype(float).values
        exog_future = block_test[ARIMA_EXOG_COLS].astype(float).values

        fitted = safe_fit_arima(y_train, exog_train, order)
        if fitted is None:
            # If fitting fails for a block, fill NaNs but keep rows
            block_out = block_test[["item_id", "timestamp", "target"]].copy()
            block_out["pred_q10"] = np.nan
            block_out["pred_q50"] = np.nan
            block_out["pred_q90"] = np.nan
            block_out["forecast_block"] = block_num
            district_pred_parts.append(block_out)

            rolling_hist = pd.concat([rolling_hist, block_test], ignore_index=True)
            rolling_hist = rolling_hist.sort_values("timestamp").reset_index(drop=True)

            start_idx = end_idx
            block_num += 1
            continue

        pred_mean, pred_lower, pred_upper = compute_prediction_intervals(
            fitted_model=fitted,
            steps=len(block_test),
            exog_future=exog_future,
            alpha=0.20   # approx 80% interval -> similar idea to q10/q90 coverage
        )

        block_out = block_test[["item_id", "timestamp", "target"]].copy()
        block_out["pred_q10"] = pred_lower
        block_out["pred_q50"] = pred_mean
        block_out["pred_q90"] = pred_upper
        block_out["forecast_block"] = block_num

        district_pred_parts.append(block_out)

        # Expand history with ACTUAL observed 2024 block
        rolling_hist = pd.concat([rolling_hist, block_test], ignore_index=True)
        rolling_hist = rolling_hist.sort_values("timestamp").reset_index(drop=True)

        start_idx = end_idx
        block_num += 1

    district_pred = pd.concat(district_pred_parts, ignore_index=True)
    district_pred = district_pred.sort_values("timestamp").reset_index(drop=True)
    district_pred["DistrictName"] = district_name
    district_pred["step_index"] = np.arange(1, len(district_pred) + 1)

    district_pred["error"] = district_pred["target"] - district_pred["pred_q50"]
    district_pred["abs_error"] = district_pred["error"].abs()
    district_pred["squared_error"] = district_pred["error"] ** 2
    district_pred["ape_percent"] = np.where(
        district_pred["target"] == 0,
        np.nan,
        (district_pred["abs_error"] / district_pred["target"].abs()) * 100
    )
    district_pred["covered_by_q10_q90"] = (
        (district_pred["target"] >= district_pred["pred_q10"]) &
        (district_pred["target"] <= district_pred["pred_q90"])
    ).astype(int)

    all_arima_predictions.append(district_pred.copy())
    all_arima_errors.append(
        district_pred[
            [
                "item_id", "DistrictName", "timestamp",
                "forecast_block", "step_index",
                "target", "pred_q10", "pred_q50", "pred_q90",
                "error", "abs_error", "squared_error",
                "ape_percent", "covered_by_q10_q90"
            ]
        ].copy()
    )

    valid_rows = district_pred["pred_q50"].notna()
    m = compute_metrics(
        district_pred.loc[valid_rows, "target"].values,
        district_pred.loc[valid_rows, "pred_q50"].values,
        district_pred.loc[valid_rows, "pred_q10"].values if valid_rows.any() else None,
        district_pred.loc[valid_rows, "pred_q90"].values if valid_rows.any() else None
    ) if valid_rows.any() else {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan, "Coverage": np.nan, "N": 0}

    arima_district_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "Order": str(order),
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "MAPE": m["MAPE"],
        "Coverage": m["Coverage"],
        "N": m["N"]
    })

arima_predictions_df = pd.concat(all_arima_predictions, ignore_index=True)
arima_errors_df = pd.concat(all_arima_errors, ignore_index=True)
arima_district_metrics_df = pd.DataFrame(arima_district_rows).sort_values("DistrictName").reset_index(drop=True)

overall_valid = arima_errors_df["pred_q50"].notna()
overall_metrics = compute_metrics(
    arima_errors_df.loc[overall_valid, "target"].values,
    arima_errors_df.loc[overall_valid, "pred_q50"].values,
    arima_errors_df.loc[overall_valid, "pred_q10"].values if overall_valid.any() else None,
    arima_errors_df.loc[overall_valid, "pred_q90"].values if overall_valid.any() else None
) if overall_valid.any() else {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan, "Coverage": np.nan, "N": 0}

overall_row = pd.DataFrame([{
    "DistrictId": "Overall",
    "DistrictName": "Overall",
    "Order": "District-specific",
    "MAE": overall_metrics["MAE"],
    "RMSE": overall_metrics["RMSE"],
    "MAPE": overall_metrics["MAPE"],
    "Coverage": overall_metrics["Coverage"],
    "N": overall_metrics["N"]
}])

final_arima_test_metrics_df = pd.concat([arima_district_metrics_df, overall_row], ignore_index=True)

# Save outputs
arima_predictions_df.to_csv(ARIMA_PREDICTIONS_CSV, index=False)
arima_errors_df.to_csv(ARIMA_STEP_ERRORS_CSV, index=False)
final_arima_test_metrics_df.to_csv(ARIMA_TEST_METRICS_CSV, index=False)

print("\nARIMA predicted rows:", len(arima_predictions_df))
print("Expected test rows:", len(test_df_sorted))

print("\nFinal ARIMA 2024 test metrics:")
display(final_arima_test_metrics_df)

print("\nSaved ARIMA files:")
print(ARIMA_VAL_RESULTS_CSV)
print(ARIMA_BEST_ORDERS_CSV)
print(ARIMA_PREDICTIONS_CSV)
print(ARIMA_STEP_ERRORS_CSV)
print(ARIMA_TEST_METRICS_CSV)

In [ ]:
# ============================================================
# LSTM training + validation residual calibration
# ============================================================
!pip install -q tensorflow scikit-learn

import os
import math
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# -----------------------------
# LSTM outputs
# -----------------------------
LSTM_SAVE_DIR = os.path.join(SAVE_DIR, "lstm_results")
os.makedirs(LSTM_SAVE_DIR, exist_ok=True)

LSTM_VAL_RESULTS_CSV = f"{LSTM_SAVE_DIR}/lstm_validation_results.csv"
LSTM_TEST_METRICS_CSV = f"{LSTM_SAVE_DIR}/lstm_test_metrics.csv"
LSTM_STEP_ERRORS_CSV = f"{LSTM_SAVE_DIR}/lstm_test_step_errors.csv"
LSTM_PREDICTIONS_CSV = f"{LSTM_SAVE_DIR}/lstm_test_predictions.csv"
LSTM_MODEL_INFO_CSV = f"{LSTM_SAVE_DIR}/lstm_model_info.csv"

# -----------------------------
# LSTM settings
# -----------------------------
LOOKBACK = 8          # use previous 8 weeks
EPOCHS = 30
BATCH_SIZE = 16
LSTM_UNITS = 32
DROPOUT = 0.1
PATIENCE = 5
LEARNING_RATE = 1e-3
SEED = 42

tf.keras.utils.set_random_seed(SEED)

# -----------------------------
# Features for LSTM
# target history + covariates
# -----------------------------
LSTM_FEATURE_COLS = ["target"] + KNOWN_COVARIATES + PAST_EXTRA_COLS

print("LSTM feature columns:", LSTM_FEATURE_COLS)
print("Lookback:", LOOKBACK)

def make_supervised_windows(df_in, feature_cols, target_col="target", lookback=8):
    """
    Convert one district dataframe into rolling supervised windows.
    X shape: (samples, lookback, num_features)
    y shape: (samples,)
    """
    df_in = df_in.sort_values("timestamp").reset_index(drop=True).copy()

    X_list, y_list, y_timestamps = [], [], []

    values = df_in[feature_cols].astype(float).values
    target_values = df_in[target_col].astype(float).values
    timestamps = df_in["timestamp"].values

    for i in range(lookback, len(df_in)):
        X_list.append(values[i - lookback:i])
        y_list.append(target_values[i])
        y_timestamps.append(timestamps[i])

    if len(X_list) == 0:
        return np.empty((0, lookback, len(feature_cols))), np.empty((0,)), np.array([])

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32), np.array(y_timestamps)

def build_lstm_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(LSTM_UNITS, dropout=DROPOUT),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="mse"
    )
    return model

def recursive_lstm_forecast(model, history_df, future_df, feature_cols, known_cov_cols, past_extra_cols, lookback,
                            x_scaler, y_scaler):
    """
    Rolling recursive forecast for a future block.
    For each step:
    - build last lookback window from current history
    - use known future covariates from future_df row
    - set unknown future past_extra features by carrying forward last available values
    - predict target
    Returns point forecasts for the future_df rows
    """
    hist = history_df.copy().sort_values("timestamp").reset_index(drop=True)
    future_df = future_df.copy().sort_values("timestamp").reset_index(drop=True)

    preds = []

    for i in range(len(future_df)):
        if len(hist) < lookback:
            raise ValueError("Not enough history to create lookback window.")

        current_row = future_df.iloc[i].copy()

        # Build synthetic next row features for prediction time
        next_features = {}

        # known future covariates come from actual future row
        for c in known_cov_cols:
            next_features[c] = float(current_row[c])

        # past-extra columns are unknown at prediction time, so carry forward last observed value
        for c in past_extra_cols:
            next_features[c] = float(hist.iloc[-1][c])

        # target placeholder not used directly in current input window row,
        # but needed after prediction when extending history
        next_features["target"] = np.nan

        # last lookback rows from history
        window_df = hist.iloc[-lookback:].copy()[["target"] + known_cov_cols + past_extra_cols]
        X_window = window_df.values.astype(np.float32)

        X_window_scaled = x_scaler.transform(X_window.reshape(-1, X_window.shape[-1])).reshape(1, lookback, -1)
        y_pred_scaled = model.predict(X_window_scaled, verbose=0).reshape(-1, 1)
        y_pred = y_scaler.inverse_transform(y_pred_scaled)[0, 0]

        preds.append(float(y_pred))

        # append predicted row to history for recursive next-step forecasting
        appended = {
            "item_id": current_row["item_id"],
            "timestamp": current_row["timestamp"],
            "target": float(y_pred),
        }
        for c in known_cov_cols:
            appended[c] = float(current_row[c])
        for c in past_extra_cols:
            appended[c] = float(hist.iloc[-1][c])

        hist = pd.concat([hist, pd.DataFrame([appended])], ignore_index=True)

    return np.array(preds, dtype=float)

lstm_artifacts = {}
lstm_val_rows = []

for item_id in common_ids:
    district_name = district_name_map.get(str(item_id), str(item_id))

    train_item = train_df[train_df["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)
    val_item = val_df[val_df["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)

    if len(train_item) <= LOOKBACK or val_item.empty:
        continue

    # build combined train+val supervised windows, then split by timestamp
    combined_tv = pd.concat([train_item, val_item], ignore_index=True).sort_values("timestamp").reset_index(drop=True)

    X_all, y_all, y_ts = make_supervised_windows(
        combined_tv,
        feature_cols=LSTM_FEATURE_COLS,
        target_col="target",
        lookback=LOOKBACK
    )

    if len(X_all) == 0:
        continue

    # train windows whose target timestamp is in train period
    train_mask = pd.to_datetime(y_ts) <= pd.to_datetime(TRAIN_END)
    val_mask = (pd.to_datetime(y_ts) >= pd.to_datetime(VAL_START)) & (pd.to_datetime(y_ts) <= pd.to_datetime(VAL_END))

    X_train = X_all[train_mask]
    y_train = y_all[train_mask]
    X_val = X_all[val_mask]
    y_val = y_all[val_mask]

    if len(X_train) == 0 or len(X_val) == 0:
        continue

    # scale features and target
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_2d = X_train.reshape(-1, X_train.shape[-1])
    X_val_2d = X_val.reshape(-1, X_val.shape[-1])

    x_scaler.fit(X_train_2d)
    X_train_scaled = x_scaler.transform(X_train_2d).reshape(X_train.shape)
    X_val_scaled = x_scaler.transform(X_val_2d).reshape(X_val.shape)

    y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).reshape(-1)
    y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).reshape(-1)

    model = build_lstm_model((LOOKBACK, len(LSTM_FEATURE_COLS)))

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_train_scaled, y_train_scaled,
        validation_data=(X_val_scaled, y_val_scaled),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=callbacks
    )

    # validation predictions for residual calibration
    val_pred_scaled = model.predict(X_val_scaled, verbose=0).reshape(-1, 1)
    val_pred = y_scaler.inverse_transform(val_pred_scaled).reshape(-1)

    val_mae = np.mean(np.abs(y_val - val_pred))
    val_rmse = math.sqrt(np.mean((y_val - val_pred) ** 2))

    residuals = y_val - val_pred
    lower_resid = np.quantile(residuals, 0.10)
    upper_resid = np.quantile(residuals, 0.90)

    lstm_artifacts[item_id] = {
        "model": model,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "lower_resid_q10": float(lower_resid),
        "upper_resid_q90": float(upper_resid),
        "epochs_trained": len(history.history["loss"]),
        "val_mae": float(val_mae),
        "val_rmse": float(val_rmse)
    }

    lstm_val_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "Validation_MAE": float(val_mae),
        "Validation_RMSE": float(val_rmse),
        "Residual_q10": float(lower_resid),
        "Residual_q90": float(upper_resid),
        "EpochsTrained": len(history.history["loss"])
    })

lstm_val_results_df = pd.DataFrame(lstm_val_rows).sort_values("DistrictName").reset_index(drop=True)
lstm_val_results_df.to_csv(LSTM_VAL_RESULTS_CSV, index=False)
lstm_val_results_df.to_csv(LSTM_MODEL_INFO_CSV, index=False)

print("\nSaved LSTM validation results:", LSTM_VAL_RESULTS_CSV)
display(lstm_val_results_df.head(20))

In [ ]:
# ============================================================
# LSTM rolling 4-step forecast on 2024
# ============================================================
if len(lstm_artifacts) == 0:
    raise RuntimeError("No LSTM models were trained successfully in Cell 5.")

history_df_lstm = pd.concat([train_df, val_df], ignore_index=True).copy()
history_df_lstm = history_df_lstm.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

test_df_sorted = test_df.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

all_lstm_predictions = []
all_lstm_errors = []
lstm_district_rows = []

for item_id in common_ids:
    if item_id not in lstm_artifacts:
        continue

    district_name = district_name_map.get(str(item_id), str(item_id))
    artifact = lstm_artifacts[item_id]

    model = artifact["model"]
    x_scaler = artifact["x_scaler"]
    y_scaler = artifact["y_scaler"]
    lower_resid = artifact["lower_resid_q10"]
    upper_resid = artifact["upper_resid_q90"]

    hist_item = history_df_lstm[history_df_lstm["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)
    test_item = test_df_sorted[test_df_sorted["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)

    if len(hist_item) < LOOKBACK or test_item.empty:
        continue

    rolling_hist = hist_item.copy()
    district_pred_parts = []

    start_idx = 0
    block_num = 1

    while start_idx < len(test_item):
        end_idx = min(start_idx + PREDICTION_LENGTH, len(test_item))
        block_test = test_item.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        pred_mean = recursive_lstm_forecast(
            model=model,
            history_df=rolling_hist,
            future_df=block_test,
            feature_cols=LSTM_FEATURE_COLS,
            known_cov_cols=KNOWN_COVARIATES,
            past_extra_cols=PAST_EXTRA_COLS,
            lookback=LOOKBACK,
            x_scaler=x_scaler,
            y_scaler=y_scaler
        )

        block_out = block_test[["item_id", "timestamp", "target"]].copy()
        block_out["pred_q50"] = pred_mean
        block_out["pred_q10"] = pred_mean + lower_resid
        block_out["pred_q90"] = pred_mean + upper_resid
        block_out["forecast_block"] = block_num

        district_pred_parts.append(block_out)

        # expand history with ACTUAL observed values from this block
        rolling_hist = pd.concat([rolling_hist, block_test], ignore_index=True)
        rolling_hist = rolling_hist.sort_values("timestamp").reset_index(drop=True)

        start_idx = end_idx
        block_num += 1

    district_pred = pd.concat(district_pred_parts, ignore_index=True)
    district_pred = district_pred.sort_values("timestamp").reset_index(drop=True)
    district_pred["DistrictName"] = district_name
    district_pred["step_index"] = np.arange(1, len(district_pred) + 1)

    district_pred["error"] = district_pred["target"] - district_pred["pred_q50"]
    district_pred["abs_error"] = district_pred["error"].abs()
    district_pred["squared_error"] = district_pred["error"] ** 2
    district_pred["ape_percent"] = np.where(
        district_pred["target"] == 0,
        np.nan,
        (district_pred["abs_error"] / district_pred["target"].abs()) * 100
    )
    district_pred["covered_by_q10_q90"] = (
        (district_pred["target"] >= district_pred["pred_q10"]) &
        (district_pred["target"] <= district_pred["pred_q90"])
    ).astype(int)

    all_lstm_predictions.append(district_pred.copy())
    all_lstm_errors.append(
        district_pred[
            [
                "item_id", "DistrictName", "timestamp",
                "forecast_block", "step_index",
                "target", "pred_q10", "pred_q50", "pred_q90",
                "error", "abs_error", "squared_error",
                "ape_percent", "covered_by_q10_q90"
            ]
        ].copy()
    )

    m = compute_metrics(
        district_pred["target"].values,
        district_pred["pred_q50"].values,
        district_pred["pred_q10"].values,
        district_pred["pred_q90"].values
    )

    lstm_district_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "MAPE": m["MAPE"],
        "Coverage": m["Coverage"],
        "N": m["N"]
    })

lstm_predictions_df = pd.concat(all_lstm_predictions, ignore_index=True)
lstm_errors_df = pd.concat(all_lstm_errors, ignore_index=True)
lstm_district_metrics_df = pd.DataFrame(lstm_district_rows).sort_values("DistrictName").reset_index(drop=True)

overall_lstm = compute_metrics(
    lstm_errors_df["target"].values,
    lstm_errors_df["pred_q50"].values,
    lstm_errors_df["pred_q10"].values,
    lstm_errors_df["pred_q90"].values
)

overall_row = pd.DataFrame([{
    "DistrictId": "Overall",
    "DistrictName": "Overall",
    "MAE": overall_lstm["MAE"],
    "RMSE": overall_lstm["RMSE"],
    "MAPE": overall_lstm["MAPE"],
    "Coverage": overall_lstm["Coverage"],
    "N": overall_lstm["N"]
}])

final_lstm_test_metrics_df = pd.concat([lstm_district_metrics_df, overall_row], ignore_index=True)

# save outputs
lstm_predictions_df.to_csv(LSTM_PREDICTIONS_CSV, index=False)
lstm_errors_df.to_csv(LSTM_STEP_ERRORS_CSV, index=False)
final_lstm_test_metrics_df.to_csv(LSTM_TEST_METRICS_CSV, index=False)

print("\nLSTM predicted rows:", len(lstm_predictions_df))
print("Expected test rows:", len(test_df_sorted))

print("\nFinal LSTM 2024 test metrics:")
display(final_lstm_test_metrics_df)

print("\nSaved LSTM files:")
print(LSTM_VAL_RESULTS_CSV)
print(LSTM_PREDICTIONS_CSV)
print(LSTM_STEP_ERRORS_CSV)
print(LSTM_TEST_METRICS_CSV)

In [ ]:
# ============================================================
# GRU training + validation residual calibration
# ============================================================
!pip install -q tensorflow scikit-learn

import os
import math
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# -----------------------------
# GRU outputs
# -----------------------------
GRU_SAVE_DIR = os.path.join(SAVE_DIR, "gru_results")
os.makedirs(GRU_SAVE_DIR, exist_ok=True)

GRU_VAL_RESULTS_CSV = f"{GRU_SAVE_DIR}/gru_validation_results.csv"
GRU_TEST_METRICS_CSV = f"{GRU_SAVE_DIR}/gru_test_metrics.csv"
GRU_STEP_ERRORS_CSV = f"{GRU_SAVE_DIR}/gru_test_step_errors.csv"
GRU_PREDICTIONS_CSV = f"{GRU_SAVE_DIR}/gru_test_predictions.csv"
GRU_MODEL_INFO_CSV = f"{GRU_SAVE_DIR}/gru_model_info.csv"

# -----------------------------
# GRU settings
# -----------------------------
GRU_LOOKBACK = 8
GRU_EPOCHS = 30
GRU_BATCH_SIZE = 16
GRU_UNITS = 32
GRU_DROPOUT = 0.1
GRU_PATIENCE = 5
GRU_LEARNING_RATE = 1e-3
GRU_SEED = 42

tf.keras.utils.set_random_seed(GRU_SEED)

# -----------------------------
# Features
# -----------------------------
GRU_FEATURE_COLS = ["target"] + KNOWN_COVARIATES + PAST_EXTRA_COLS

print("GRU feature columns:", GRU_FEATURE_COLS)
print("GRU lookback:", GRU_LOOKBACK)

def make_supervised_windows(df_in, feature_cols, target_col="target", lookback=8):
    df_in = df_in.sort_values("timestamp").reset_index(drop=True).copy()

    X_list, y_list, y_timestamps = [], [], []

    values = df_in[feature_cols].astype(float).values
    target_values = df_in[target_col].astype(float).values
    timestamps = df_in["timestamp"].values

    for i in range(lookback, len(df_in)):
        X_list.append(values[i - lookback:i])
        y_list.append(target_values[i])
        y_timestamps.append(timestamps[i])

    if len(X_list) == 0:
        return np.empty((0, lookback, len(feature_cols))), np.empty((0,)), np.array([])

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32), np.array(y_timestamps)

def build_gru_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.GRU(GRU_UNITS, dropout=GRU_DROPOUT),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=GRU_LEARNING_RATE),
        loss="mse"
    )
    return model

def recursive_gru_forecast(model, history_df, future_df, known_cov_cols, past_extra_cols, lookback,
                           x_scaler, y_scaler):
    """
    Recursive multi-step GRU forecast for a future block.
    """
    hist = history_df.copy().sort_values("timestamp").reset_index(drop=True)
    future_df = future_df.copy().sort_values("timestamp").reset_index(drop=True)

    preds = []

    for i in range(len(future_df)):
        if len(hist) < lookback:
            raise ValueError("Not enough history to create lookback window.")

        current_row = future_df.iloc[i].copy()

        # Build window from latest history only
        window_df = hist.iloc[-lookback:].copy()[["target"] + known_cov_cols + past_extra_cols]
        X_window = window_df.values.astype(np.float32)

        X_window_scaled = x_scaler.transform(
            X_window.reshape(-1, X_window.shape[-1])
        ).reshape(1, lookback, -1)

        y_pred_scaled = model.predict(X_window_scaled, verbose=0).reshape(-1, 1)
        y_pred = y_scaler.inverse_transform(y_pred_scaled)[0, 0]

        preds.append(float(y_pred))

        # append predicted step so next step can roll forward
        appended = {
            "item_id": current_row["item_id"],
            "timestamp": current_row["timestamp"],
            "target": float(y_pred),
        }

        for c in known_cov_cols:
            appended[c] = float(current_row[c])

        # carry forward latest available past-extra features
        for c in past_extra_cols:
            appended[c] = float(hist.iloc[-1][c])

        hist = pd.concat([hist, pd.DataFrame([appended])], ignore_index=True)

    return np.array(preds, dtype=float)

gru_artifacts = {}
gru_val_rows = []

for item_id in common_ids:
    district_name = district_name_map.get(str(item_id), str(item_id))

    train_item = train_df[train_df["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)
    val_item = val_df[val_df["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)

    if len(train_item) <= GRU_LOOKBACK or val_item.empty:
        continue

    combined_tv = pd.concat([train_item, val_item], ignore_index=True).sort_values("timestamp").reset_index(drop=True)

    X_all, y_all, y_ts = make_supervised_windows(
        combined_tv,
        feature_cols=GRU_FEATURE_COLS,
        target_col="target",
        lookback=GRU_LOOKBACK
    )

    if len(X_all) == 0:
        continue

    train_mask = pd.to_datetime(y_ts) <= pd.to_datetime(TRAIN_END)
    val_mask = (pd.to_datetime(y_ts) >= pd.to_datetime(VAL_START)) & (pd.to_datetime(y_ts) <= pd.to_datetime(VAL_END))

    X_train = X_all[train_mask]
    y_train = y_all[train_mask]
    X_val = X_all[val_mask]
    y_val = y_all[val_mask]

    if len(X_train) == 0 or len(X_val) == 0:
        continue

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_2d = X_train.reshape(-1, X_train.shape[-1])
    X_val_2d = X_val.reshape(-1, X_val.shape[-1])

    x_scaler.fit(X_train_2d)
    X_train_scaled = x_scaler.transform(X_train_2d).reshape(X_train.shape)
    X_val_scaled = x_scaler.transform(X_val_2d).reshape(X_val.shape)

    y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).reshape(-1)
    y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).reshape(-1)

    model = build_gru_model((GRU_LOOKBACK, len(GRU_FEATURE_COLS)))

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=GRU_PATIENCE,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_train_scaled, y_train_scaled,
        validation_data=(X_val_scaled, y_val_scaled),
        epochs=GRU_EPOCHS,
        batch_size=GRU_BATCH_SIZE,
        verbose=0,
        callbacks=callbacks
    )

    val_pred_scaled = model.predict(X_val_scaled, verbose=0).reshape(-1, 1)
    val_pred = y_scaler.inverse_transform(val_pred_scaled).reshape(-1)

    val_mae = np.mean(np.abs(y_val - val_pred))
    val_rmse = math.sqrt(np.mean((y_val - val_pred) ** 2))

    residuals = y_val - val_pred
    lower_resid = np.quantile(residuals, 0.10)
    upper_resid = np.quantile(residuals, 0.90)

    gru_artifacts[item_id] = {
        "model": model,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "lower_resid_q10": float(lower_resid),
        "upper_resid_q90": float(upper_resid),
        "epochs_trained": len(history.history["loss"]),
        "val_mae": float(val_mae),
        "val_rmse": float(val_rmse)
    }

    gru_val_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "Validation_MAE": float(val_mae),
        "Validation_RMSE": float(val_rmse),
        "Residual_q10": float(lower_resid),
        "Residual_q90": float(upper_resid),
        "EpochsTrained": len(history.history["loss"])
    })

gru_val_results_df = pd.DataFrame(gru_val_rows).sort_values("DistrictName").reset_index(drop=True)
gru_val_results_df.to_csv(GRU_VAL_RESULTS_CSV, index=False)
gru_val_results_df.to_csv(GRU_MODEL_INFO_CSV, index=False)

print("\nSaved GRU validation results:", GRU_VAL_RESULTS_CSV)
display(gru_val_results_df.head(20))

In [ ]:
# ============================================================
# GRU rolling 4-step forecast on 2024
# ============================================================
if len(gru_artifacts) == 0:
    raise RuntimeError("No GRU models were trained successfully in Cell 7.")

history_df_gru = pd.concat([train_df, val_df], ignore_index=True).copy()
history_df_gru = history_df_gru.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

test_df_sorted = test_df.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

all_gru_predictions = []
all_gru_errors = []
gru_district_rows = []

for item_id in common_ids:
    if item_id not in gru_artifacts:
        continue

    district_name = district_name_map.get(str(item_id), str(item_id))
    artifact = gru_artifacts[item_id]

    model = artifact["model"]
    x_scaler = artifact["x_scaler"]
    y_scaler = artifact["y_scaler"]
    lower_resid = artifact["lower_resid_q10"]
    upper_resid = artifact["upper_resid_q90"]

    hist_item = history_df_gru[history_df_gru["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)
    test_item = test_df_sorted[test_df_sorted["item_id"] == item_id].copy().sort_values("timestamp").reset_index(drop=True)

    if len(hist_item) < GRU_LOOKBACK or test_item.empty:
        continue

    rolling_hist = hist_item.copy()
    district_pred_parts = []

    start_idx = 0
    block_num = 1

    while start_idx < len(test_item):
        end_idx = min(start_idx + PREDICTION_LENGTH, len(test_item))
        block_test = test_item.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        pred_mean = recursive_gru_forecast(
            model=model,
            history_df=rolling_hist,
            future_df=block_test,
            known_cov_cols=KNOWN_COVARIATES,
            past_extra_cols=PAST_EXTRA_COLS,
            lookback=GRU_LOOKBACK,
            x_scaler=x_scaler,
            y_scaler=y_scaler
        )

        block_out = block_test[["item_id", "timestamp", "target"]].copy()
        block_out["pred_q50"] = pred_mean
        block_out["pred_q10"] = pred_mean + lower_resid
        block_out["pred_q90"] = pred_mean + upper_resid
        block_out["forecast_block"] = block_num

        district_pred_parts.append(block_out)

        # expand history with ACTUAL observed values from this block
        rolling_hist = pd.concat([rolling_hist, block_test], ignore_index=True)
        rolling_hist = rolling_hist.sort_values("timestamp").reset_index(drop=True)

        start_idx = end_idx
        block_num += 1

    district_pred = pd.concat(district_pred_parts, ignore_index=True)
    district_pred = district_pred.sort_values("timestamp").reset_index(drop=True)
    district_pred["DistrictName"] = district_name
    district_pred["step_index"] = np.arange(1, len(district_pred) + 1)

    district_pred["error"] = district_pred["target"] - district_pred["pred_q50"]
    district_pred["abs_error"] = district_pred["error"].abs()
    district_pred["squared_error"] = district_pred["error"] ** 2
    district_pred["ape_percent"] = np.where(
        district_pred["target"] == 0,
        np.nan,
        (district_pred["abs_error"] / district_pred["target"].abs()) * 100
    )
    district_pred["covered_by_q10_q90"] = (
        (district_pred["target"] >= district_pred["pred_q10"]) &
        (district_pred["target"] <= district_pred["pred_q90"])
    ).astype(int)

    all_gru_predictions.append(district_pred.copy())
    all_gru_errors.append(
        district_pred[
            [
                "item_id", "DistrictName", "timestamp",
                "forecast_block", "step_index",
                "target", "pred_q10", "pred_q50", "pred_q90",
                "error", "abs_error", "squared_error",
                "ape_percent", "covered_by_q10_q90"
            ]
        ].copy()
    )

    m = compute_metrics(
        district_pred["target"].values,
        district_pred["pred_q50"].values,
        district_pred["pred_q10"].values,
        district_pred["pred_q90"].values
    )

    gru_district_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "MAPE": m["MAPE"],
        "Coverage": m["Coverage"],
        "N": m["N"]
    })

gru_predictions_df = pd.concat(all_gru_predictions, ignore_index=True)
gru_errors_df = pd.concat(all_gru_errors, ignore_index=True)
gru_district_metrics_df = pd.DataFrame(gru_district_rows).sort_values("DistrictName").reset_index(drop=True)

overall_gru = compute_metrics(
    gru_errors_df["target"].values,
    gru_errors_df["pred_q50"].values,
    gru_errors_df["pred_q10"].values,
    gru_errors_df["pred_q90"].values
)

overall_row = pd.DataFrame([{
    "DistrictId": "Overall",
    "DistrictName": "Overall",
    "MAE": overall_gru["MAE"],
    "RMSE": overall_gru["RMSE"],
    "MAPE": overall_gru["MAPE"],
    "Coverage": overall_gru["Coverage"],
    "N": overall_gru["N"]
}])

final_gru_test_metrics_df = pd.concat([gru_district_metrics_df, overall_row], ignore_index=True)

# save outputs
gru_predictions_df.to_csv(GRU_PREDICTIONS_CSV, index=False)
gru_errors_df.to_csv(GRU_STEP_ERRORS_CSV, index=False)
final_gru_test_metrics_df.to_csv(GRU_TEST_METRICS_CSV, index=False)

print("\nGRU predicted rows:", len(gru_predictions_df))
print("Expected test rows:", len(test_df_sorted))

print("\nFinal GRU 2024 test metrics:")
display(final_gru_test_metrics_df)

print("\nSaved GRU files:")
print(GRU_VAL_RESULTS_CSV)
print(GRU_PREDICTIONS_CSV)
print(GRU_STEP_ERRORS_CSV)
print(GRU_TEST_METRICS_CSV)

In [ ]:
# ============================================================
# TFT validation search
# ============================================================
!pip install -q neuralforecast

import os
import math
import warnings
import numpy as np
import pandas as pd

from neuralforecast import NeuralForecast
from neuralforecast.models import TFT

warnings.filterwarnings("ignore")

# -----------------------------
# TFT outputs
# -----------------------------
TFT_SAVE_DIR = os.path.join(SAVE_DIR, "tft_results")
os.makedirs(TFT_SAVE_DIR, exist_ok=True)

TFT_VAL_RESULTS_CSV = f"{TFT_SAVE_DIR}/tft_validation_results.csv"
TFT_TEST_METRICS_CSV = f"{TFT_SAVE_DIR}/tft_test_metrics.csv"
TFT_STEP_ERRORS_CSV = f"{TFT_SAVE_DIR}/tft_test_step_errors.csv"
TFT_PREDICTIONS_CSV = f"{TFT_SAVE_DIR}/tft_test_predictions.csv"
TFT_BEST_CONFIGS_CSV = f"{TFT_SAVE_DIR}/tft_best_configs.csv"

# -----------------------------
# TFT exogenous lists
# -----------------------------
TFT_FUTR_EXOG = KNOWN_COVARIATES.copy()
TFT_HIST_EXOG = PAST_EXTRA_COLS.copy()

# -----------------------------
# TFT config grid
# modest grid so runtime stays manageable
# -----------------------------
TFT_CONFIG_GRID = [
    {"input_size": 8,  "max_steps": 100, "learning_rate": 1e-3, "hidden_size": 16},
    {"input_size": 12, "max_steps": 100, "learning_rate": 1e-3, "hidden_size": 16},
    {"input_size": 8,  "max_steps": 150, "learning_rate": 5e-4, "hidden_size": 32},
    {"input_size": 12, "max_steps": 150, "learning_rate": 5e-4, "hidden_size": 32},
]

print("TFT future exogenous:", TFT_FUTR_EXOG)
print("TFT historic exogenous:", TFT_HIST_EXOG)
print("Candidate TFT configs:")
for cfg in TFT_CONFIG_GRID:
    print(cfg)

def fit_tft_and_forecast(train_hist_df, future_block_df, config):
    """
    Fit TFT on one district history and forecast the next len(future_block_df) steps.
    Returns mean forecast array or None on failure.
    """
    train_nf = train_hist_df.copy()
    future_nf = future_block_df.copy()

    h = len(future_nf)

    try:
        model = TFT(
            h=h,
            input_size=config["input_size"],
            hidden_size=config["hidden_size"],
            max_steps=config["max_steps"],
            learning_rate=config["learning_rate"],
            futr_exog_list=TFT_FUTR_EXOG,
            hist_exog_list=TFT_HIST_EXOG,
            random_seed=42,
        )

        nf = NeuralForecast(models=[model], freq="W-MON")
        nf.fit(df=train_nf)

        # NeuralForecast expects future dataframe with unique_id, ds and future exogenous columns
        futr_df = future_nf[["unique_id", "ds"] + TFT_FUTR_EXOG].copy()
        fcst = nf.predict(futr_df=futr_df)

        pred_col = [c for c in fcst.columns if c not in ["unique_id", "ds"]][0]
        return fcst[pred_col].astype(float).values
    except Exception:
        return None

def rolling_validate_tft(train_item_df, val_item_df, config, prediction_length=4):
    """
    Rolling 4-step validation on 2023.
    History expands with actual validation blocks.
    """
    rolling_hist = train_item_df.copy().sort_values("ds").reset_index(drop=True)
    val_item = val_item_df.copy().sort_values("ds").reset_index(drop=True)

    preds = []
    actuals = []

    start_idx = 0
    while start_idx < len(val_item):
        end_idx = min(start_idx + prediction_length, len(val_item))
        block_val = val_item.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        pred = fit_tft_and_forecast(
            train_hist_df=rolling_hist,
            future_block_df=block_val,
            config=config
        )
        if pred is None:
            return np.inf

        preds.extend(pred.tolist())
        actuals.extend(block_val["y"].astype(float).tolist())

        # Expand history with ACTUAL validation block
        rolling_hist = pd.concat([rolling_hist, block_val], ignore_index=True)
        rolling_hist = rolling_hist.sort_values("ds").reset_index(drop=True)

        start_idx = end_idx

    preds = np.asarray(preds, dtype=float)
    actuals = np.asarray(actuals, dtype=float)
    mae = np.mean(np.abs(actuals - preds))
    return mae

# -----------------------------
# Prepare TFT-formatted data
# NeuralForecast expects: unique_id, ds, y
# Plus exogenous columns
# -----------------------------
train_tft_df = train_df.rename(columns={"item_id": "unique_id", "timestamp": "ds", "target": "y"}).copy()
val_tft_df   = val_df.rename(columns={"item_id": "unique_id", "timestamp": "ds", "target": "y"}).copy()
test_tft_df  = test_df.rename(columns={"item_id": "unique_id", "timestamp": "ds", "target": "y"}).copy()

tft_validation_rows = []
tft_best_rows = []

for item_id in common_ids:
    district_name = district_name_map.get(str(item_id), str(item_id))

    train_item = train_tft_df[train_tft_df["unique_id"] == item_id].copy().sort_values("ds").reset_index(drop=True)
    val_item = val_tft_df[val_tft_df["unique_id"] == item_id].copy().sort_values("ds").reset_index(drop=True)

    if train_item.empty or val_item.empty:
        continue

    best_cfg = None
    best_mae = np.inf

    for cfg in TFT_CONFIG_GRID:
        val_mae = rolling_validate_tft(
            train_item_df=train_item,
            val_item_df=val_item,
            config=cfg,
            prediction_length=PREDICTION_LENGTH
        )

        tft_validation_rows.append({
            "DistrictId": item_id,
            "DistrictName": district_name,
            "input_size": cfg["input_size"],
            "hidden_size": cfg["hidden_size"],
            "max_steps": cfg["max_steps"],
            "learning_rate": cfg["learning_rate"],
            "Validation_MAE": val_mae
        })

        if np.isfinite(val_mae) and val_mae < best_mae:
            best_mae = val_mae
            best_cfg = cfg.copy()

    tft_best_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "Best_input_size": None if best_cfg is None else best_cfg["input_size"],
        "Best_hidden_size": None if best_cfg is None else best_cfg["hidden_size"],
        "Best_max_steps": None if best_cfg is None else best_cfg["max_steps"],
        "Best_learning_rate": None if best_cfg is None else best_cfg["learning_rate"],
        "BestValidationMAE": best_mae
    })

tft_val_results_df = pd.DataFrame(tft_validation_rows)
tft_best_configs_df = pd.DataFrame(tft_best_rows)

tft_val_results_df.to_csv(TFT_VAL_RESULTS_CSV, index=False)
tft_best_configs_df.to_csv(TFT_BEST_CONFIGS_CSV, index=False)

print("\nSaved TFT validation search:", TFT_VAL_RESULTS_CSV)
print("Saved TFT best configs:", TFT_BEST_CONFIGS_CSV)
display(tft_best_configs_df.head(20))

In [ ]:
# ============================================================
# TFT rolling 4-step forecast on 2024
# ============================================================
def compute_tft_intervals_from_validation_mae(pred_mean, val_mae):
    """
    Approximate q10/q90 interval using validation MAE.
    Keeps output structure parallel to the other baselines.
    """
    pred_mean = np.asarray(pred_mean, dtype=float)
    width = 1.64 * float(val_mae)
    lower = pred_mean - width
    upper = pred_mean + width
    return lower, upper

# lookup best configs
tft_best_lookup = {}
for _, row in tft_best_configs_df.iterrows():
    item_id = str(row["DistrictId"])
    if pd.notna(row["Best_input_size"]):
        tft_best_lookup[item_id] = {
            "input_size": int(row["Best_input_size"]),
            "hidden_size": int(row["Best_hidden_size"]),
            "max_steps": int(row["Best_max_steps"]),
            "learning_rate": float(row["Best_learning_rate"]),
            "val_mae": float(row["BestValidationMAE"]),
        }

print("Districts with selected TFT config:", len(tft_best_lookup))

history_tft_df = pd.concat([train_tft_df, val_tft_df], ignore_index=True).copy()
history_tft_df = history_tft_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

test_tft_df_sorted = test_tft_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

all_tft_predictions = []
all_tft_errors = []
tft_district_rows = []

for item_id in common_ids:
    if item_id not in tft_best_lookup:
        continue

    district_name = district_name_map.get(str(item_id), str(item_id))
    cfg = tft_best_lookup[item_id]

    hist_item = history_tft_df[history_tft_df["unique_id"] == item_id].copy().sort_values("ds").reset_index(drop=True)
    test_item = test_tft_df_sorted[test_tft_df_sorted["unique_id"] == item_id].copy().sort_values("ds").reset_index(drop=True)

    if hist_item.empty or test_item.empty:
        continue

    rolling_hist = hist_item.copy()
    district_pred_parts = []

    start_idx = 0
    block_num = 1

    while start_idx < len(test_item):
        end_idx = min(start_idx + PREDICTION_LENGTH, len(test_item))
        block_test = test_item.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        pred_mean = fit_tft_and_forecast(
            train_hist_df=rolling_hist,
            future_block_df=block_test,
            config=cfg
        )

        if pred_mean is None:
            block_out = block_test[["unique_id", "ds", "y"]].copy()
            block_out["pred_q10"] = np.nan
            block_out["pred_q50"] = np.nan
            block_out["pred_q90"] = np.nan
            block_out["forecast_block"] = block_num
        else:
            pred_q10, pred_q90 = compute_tft_intervals_from_validation_mae(pred_mean, cfg["val_mae"])

            block_out = block_test[["unique_id", "ds", "y"]].copy()
            block_out["pred_q10"] = pred_q10
            block_out["pred_q50"] = pred_mean
            block_out["pred_q90"] = pred_q90
            block_out["forecast_block"] = block_num

        district_pred_parts.append(block_out)

        # Expand history with ACTUAL observed test block
        rolling_hist = pd.concat([rolling_hist, block_test], ignore_index=True)
        rolling_hist = rolling_hist.sort_values("ds").reset_index(drop=True)

        start_idx = end_idx
        block_num += 1

    district_pred = pd.concat(district_pred_parts, ignore_index=True)
    district_pred = district_pred.sort_values("ds").reset_index(drop=True)
    district_pred["DistrictName"] = district_name
    district_pred["step_index"] = np.arange(1, len(district_pred) + 1)

    district_pred = district_pred.rename(columns={
        "unique_id": "item_id",
        "ds": "timestamp",
        "y": "target"
    })

    district_pred["error"] = district_pred["target"] - district_pred["pred_q50"]
    district_pred["abs_error"] = district_pred["error"].abs()
    district_pred["squared_error"] = district_pred["error"] ** 2
    district_pred["ape_percent"] = np.where(
        district_pred["target"] == 0,
        np.nan,
        (district_pred["abs_error"] / district_pred["target"].abs()) * 100
    )
    district_pred["covered_by_q10_q90"] = (
        (district_pred["target"] >= district_pred["pred_q10"]) &
        (district_pred["target"] <= district_pred["pred_q90"])
    ).astype(int)

    all_tft_predictions.append(district_pred.copy())
    all_tft_errors.append(
        district_pred[
            [
                "item_id", "DistrictName", "timestamp",
                "forecast_block", "step_index",
                "target", "pred_q10", "pred_q50", "pred_q90",
                "error", "abs_error", "squared_error",
                "ape_percent", "covered_by_q10_q90"
            ]
        ].copy()
    )

    valid_rows = district_pred["pred_q50"].notna()
    m = compute_metrics(
        district_pred.loc[valid_rows, "target"].values,
        district_pred.loc[valid_rows, "pred_q50"].values,
        district_pred.loc[valid_rows, "pred_q10"].values if valid_rows.any() else None,
        district_pred.loc[valid_rows, "pred_q90"].values if valid_rows.any() else None
    ) if valid_rows.any() else {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan, "Coverage": np.nan, "N": 0}

    tft_district_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "input_size": cfg["input_size"],
        "hidden_size": cfg["hidden_size"],
        "max_steps": cfg["max_steps"],
        "learning_rate": cfg["learning_rate"],
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "MAPE": m["MAPE"],
        "Coverage": m["Coverage"],
        "N": m["N"]
    })

tft_predictions_df = pd.concat(all_tft_predictions, ignore_index=True)
tft_errors_df = pd.concat(all_tft_errors, ignore_index=True)
tft_district_metrics_df = pd.DataFrame(tft_district_rows).sort_values("DistrictName").reset_index(drop=True)

overall_valid = tft_errors_df["pred_q50"].notna()
overall_metrics = compute_metrics(
    tft_errors_df.loc[overall_valid, "target"].values,
    tft_errors_df.loc[overall_valid, "pred_q50"].values,
    tft_errors_df.loc[overall_valid, "pred_q10"].values if overall_valid.any() else None,
    tft_errors_df.loc[overall_valid, "pred_q90"].values if overall_valid.any() else None
) if overall_valid.any() else {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan, "Coverage": np.nan, "N": 0}

overall_row = pd.DataFrame([{
    "DistrictId": "Overall",
    "DistrictName": "Overall",
    "input_size": "district-specific",
    "hidden_size": "district-specific",
    "max_steps": "district-specific",
    "learning_rate": "district-specific",
    "MAE": overall_metrics["MAE"],
    "RMSE": overall_metrics["RMSE"],
    "MAPE": overall_metrics["MAPE"],
    "Coverage": overall_metrics["Coverage"],
    "N": overall_metrics["N"]
}])

final_tft_test_metrics_df = pd.concat([tft_district_metrics_df, overall_row], ignore_index=True)

# save outputs
tft_predictions_df.to_csv(TFT_PREDICTIONS_CSV, index=False)
tft_errors_df.to_csv(TFT_STEP_ERRORS_CSV, index=False)
final_tft_test_metrics_df.to_csv(TFT_TEST_METRICS_CSV, index=False)

print("\nTFT predicted rows:", len(tft_predictions_df))
print("Expected test rows:", len(test_tft_df_sorted))

print("\nFinal TFT 2024 test metrics:")
display(final_tft_test_metrics_df)

print("\nSaved TFT files:")
print(TFT_VAL_RESULTS_CSV)
print(TFT_BEST_CONFIGS_CSV)
print(TFT_PREDICTIONS_CSV)
print(TFT_STEP_ERRORS_CSV)
print(TFT_TEST_METRICS_CSV)

In [ ]:
# ============================================================
# Prophet validation search
# ============================================================
!pip install -q prophet

import os
import math
import warnings
import numpy as np
import pandas as pd

from prophet import Prophet

warnings.filterwarnings("ignore")

# -----------------------------
# Prophet outputs
# -----------------------------
PROPHET_SAVE_DIR = os.path.join(SAVE_DIR, "prophet_results")
os.makedirs(PROPHET_SAVE_DIR, exist_ok=True)

PROPHET_VAL_RESULTS_CSV = f"{PROPHET_SAVE_DIR}/prophet_validation_results.csv"
PROPHET_TEST_METRICS_CSV = f"{PROPHET_SAVE_DIR}/prophet_test_metrics.csv"
PROPHET_STEP_ERRORS_CSV = f"{PROPHET_SAVE_DIR}/prophet_test_step_errors.csv"
PROPHET_PREDICTIONS_CSV = f"{PROPHET_SAVE_DIR}/prophet_test_predictions.csv"
PROPHET_BEST_CONFIGS_CSV = f"{PROPHET_SAVE_DIR}/prophet_best_configs.csv"

# -----------------------------
# Prophet regressors
# Use future-known covariates only
# -----------------------------
PROPHET_REGRESSORS = KNOWN_COVARIATES.copy()

# -----------------------------
# Prophet config grid
# modest grid for manageable runtime
# -----------------------------
PROPHET_CONFIG_GRID = [
    {"changepoint_prior_scale": 0.05, "seasonality_prior_scale": 10.0, "seasonality_mode": "additive"},
    {"changepoint_prior_scale": 0.10, "seasonality_prior_scale": 10.0, "seasonality_mode": "additive"},
    {"changepoint_prior_scale": 0.05, "seasonality_prior_scale": 5.0,  "seasonality_mode": "multiplicative"},
    {"changepoint_prior_scale": 0.10, "seasonality_prior_scale": 5.0,  "seasonality_mode": "multiplicative"},
]

print("Prophet regressors:", PROPHET_REGRESSORS)
print("Candidate Prophet configs:")
for cfg in PROPHET_CONFIG_GRID:
    print(cfg)

def fit_prophet_and_forecast(train_hist_df, future_block_df, config):
    """
    Fit Prophet on one district history and forecast the next len(future_block_df) steps.
    Returns dataframe with ds, yhat, yhat_lower, yhat_upper or None on failure.
    """
    train_pf = train_hist_df[["ds", "y"] + PROPHET_REGRESSORS].copy()
    future_pf = future_block_df[["ds"] + PROPHET_REGRESSORS].copy()

    try:
        m = Prophet(
            changepoint_prior_scale=config["changepoint_prior_scale"],
            seasonality_prior_scale=config["seasonality_prior_scale"],
            seasonality_mode=config["seasonality_mode"],
            interval_width=0.80,  # roughly analogous to q10-q90 style interval
            weekly_seasonality=True,
            yearly_seasonality=True,
            daily_seasonality=False,
        )

        for reg in PROPHET_REGRESSORS:
            m.add_regressor(reg)

        m.fit(train_pf)

        forecast = m.predict(future_pf)
        return forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()

    except Exception:
        return None

def rolling_validate_prophet(train_item_df, val_item_df, config, prediction_length=4):
    """
    Rolling 4-step validation on 2023.
    History expands with actual validation blocks.
    """
    rolling_hist = train_item_df.copy().sort_values("ds").reset_index(drop=True)
    val_item = val_item_df.copy().sort_values("ds").reset_index(drop=True)

    preds = []
    actuals = []

    start_idx = 0
    while start_idx < len(val_item):
        end_idx = min(start_idx + prediction_length, len(val_item))
        block_val = val_item.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        forecast = fit_prophet_and_forecast(
            train_hist_df=rolling_hist,
            future_block_df=block_val,
            config=config
        )
        if forecast is None:
            return np.inf

        preds.extend(forecast["yhat"].astype(float).tolist())
        actuals.extend(block_val["y"].astype(float).tolist())

        # Expand history with ACTUAL validation block
        rolling_hist = pd.concat([rolling_hist, block_val], ignore_index=True)
        rolling_hist = rolling_hist.sort_values("ds").reset_index(drop=True)

        start_idx = end_idx

    preds = np.asarray(preds, dtype=float)
    actuals = np.asarray(actuals, dtype=float)
    mae = np.mean(np.abs(actuals - preds))
    return mae

# -----------------------------
# Prepare Prophet-formatted data
# Prophet expects ds, y
# Plus regressor columns
# -----------------------------
train_prophet_df = train_df.rename(columns={"item_id": "unique_id", "timestamp": "ds", "target": "y"}).copy()
val_prophet_df   = val_df.rename(columns={"item_id": "unique_id", "timestamp": "ds", "target": "y"}).copy()
test_prophet_df  = test_df.rename(columns={"item_id": "unique_id", "timestamp": "ds", "target": "y"}).copy()

prophet_validation_rows = []
prophet_best_rows = []

for item_id in common_ids:
    district_name = district_name_map.get(str(item_id), str(item_id))

    train_item = train_prophet_df[train_prophet_df["unique_id"] == item_id].copy().sort_values("ds").reset_index(drop=True)
    val_item = val_prophet_df[val_prophet_df["unique_id"] == item_id].copy().sort_values("ds").reset_index(drop=True)

    if train_item.empty or val_item.empty:
        continue

    best_cfg = None
    best_mae = np.inf

    for cfg in PROPHET_CONFIG_GRID:
        val_mae = rolling_validate_prophet(
            train_item_df=train_item,
            val_item_df=val_item,
            config=cfg,
            prediction_length=PREDICTION_LENGTH
        )

        prophet_validation_rows.append({
            "DistrictId": item_id,
            "DistrictName": district_name,
            "changepoint_prior_scale": cfg["changepoint_prior_scale"],
            "seasonality_prior_scale": cfg["seasonality_prior_scale"],
            "seasonality_mode": cfg["seasonality_mode"],
            "Validation_MAE": val_mae
        })

        if np.isfinite(val_mae) and val_mae < best_mae:
            best_mae = val_mae
            best_cfg = cfg.copy()

    prophet_best_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "Best_changepoint_prior_scale": None if best_cfg is None else best_cfg["changepoint_prior_scale"],
        "Best_seasonality_prior_scale": None if best_cfg is None else best_cfg["seasonality_prior_scale"],
        "Best_seasonality_mode": None if best_cfg is None else best_cfg["seasonality_mode"],
        "BestValidationMAE": best_mae
    })

prophet_val_results_df = pd.DataFrame(prophet_validation_rows)
prophet_best_configs_df = pd.DataFrame(prophet_best_rows)

prophet_val_results_df.to_csv(PROPHET_VAL_RESULTS_CSV, index=False)
prophet_best_configs_df.to_csv(PROPHET_BEST_CONFIGS_CSV, index=False)

print("\nSaved Prophet validation search:", PROPHET_VAL_RESULTS_CSV)
print("Saved Prophet best configs:", PROPHET_BEST_CONFIGS_CSV)
display(prophet_best_configs_df.head(20))

In [ ]:
# ============================================================
# Prophet rolling 4-step forecast on 2024
# ============================================================
# lookup best configs
prophet_best_lookup = {}
for _, row in prophet_best_configs_df.iterrows():
    item_id = str(row["DistrictId"])
    if pd.notna(row["Best_changepoint_prior_scale"]):
        prophet_best_lookup[item_id] = {
            "changepoint_prior_scale": float(row["Best_changepoint_prior_scale"]),
            "seasonality_prior_scale": float(row["Best_seasonality_prior_scale"]),
            "seasonality_mode": str(row["Best_seasonality_mode"]),
            "val_mae": float(row["BestValidationMAE"]),
        }

print("Districts with selected Prophet config:", len(prophet_best_lookup))

history_prophet_df = pd.concat([train_prophet_df, val_prophet_df], ignore_index=True).copy()
history_prophet_df = history_prophet_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

test_prophet_df_sorted = test_prophet_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

all_prophet_predictions = []
all_prophet_errors = []
prophet_district_rows = []

for item_id in common_ids:
    if item_id not in prophet_best_lookup:
        continue

    district_name = district_name_map.get(str(item_id), str(item_id))
    cfg = prophet_best_lookup[item_id]

    hist_item = history_prophet_df[history_prophet_df["unique_id"] == item_id].copy().sort_values("ds").reset_index(drop=True)
    test_item = test_prophet_df_sorted[test_prophet_df_sorted["unique_id"] == item_id].copy().sort_values("ds").reset_index(drop=True)

    if hist_item.empty or test_item.empty:
        continue

    rolling_hist = hist_item.copy()
    district_pred_parts = []

    start_idx = 0
    block_num = 1

    while start_idx < len(test_item):
        end_idx = min(start_idx + PREDICTION_LENGTH, len(test_item))
        block_test = test_item.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        forecast = fit_prophet_and_forecast(
            train_hist_df=rolling_hist,
            future_block_df=block_test,
            config=cfg
        )

        if forecast is None:
            block_out = block_test[["unique_id", "ds", "y"]].copy()
            block_out["pred_q10"] = np.nan
            block_out["pred_q50"] = np.nan
            block_out["pred_q90"] = np.nan
            block_out["forecast_block"] = block_num
        else:
            block_out = block_test[["unique_id", "ds", "y"]].copy()
            block_out = block_out.merge(forecast, on="ds", how="left")
            block_out["pred_q10"] = block_out["yhat_lower"]
            block_out["pred_q50"] = block_out["yhat"]
            block_out["pred_q90"] = block_out["yhat_upper"]
            block_out["forecast_block"] = block_num
            block_out = block_out.drop(columns=["yhat", "yhat_lower", "yhat_upper"])

        district_pred_parts.append(block_out)

        # Expand history with ACTUAL observed test block
        rolling_hist = pd.concat([rolling_hist, block_test], ignore_index=True)
        rolling_hist = rolling_hist.sort_values("ds").reset_index(drop=True)

        start_idx = end_idx
        block_num += 1

    district_pred = pd.concat(district_pred_parts, ignore_index=True)
    district_pred = district_pred.sort_values("ds").reset_index(drop=True)
    district_pred["DistrictName"] = district_name
    district_pred["step_index"] = np.arange(1, len(district_pred) + 1)

    district_pred = district_pred.rename(columns={
        "unique_id": "item_id",
        "ds": "timestamp",
        "y": "target"
    })

    district_pred["error"] = district_pred["target"] - district_pred["pred_q50"]
    district_pred["abs_error"] = district_pred["error"].abs()
    district_pred["squared_error"] = district_pred["error"] ** 2
    district_pred["ape_percent"] = np.where(
        district_pred["target"] == 0,
        np.nan,
        (district_pred["abs_error"] / district_pred["target"].abs()) * 100
    )
    district_pred["covered_by_q10_q90"] = (
        (district_pred["target"] >= district_pred["pred_q10"]) &
        (district_pred["target"] <= district_pred["pred_q90"])
    ).astype(int)

    all_prophet_predictions.append(district_pred.copy())
    all_prophet_errors.append(
        district_pred[
            [
                "item_id", "DistrictName", "timestamp",
                "forecast_block", "step_index",
                "target", "pred_q10", "pred_q50", "pred_q90",
                "error", "abs_error", "squared_error",
                "ape_percent", "covered_by_q10_q90"
            ]
        ].copy()
    )

    valid_rows = district_pred["pred_q50"].notna()
    m = compute_metrics(
        district_pred.loc[valid_rows, "target"].values,
        district_pred.loc[valid_rows, "pred_q50"].values,
        district_pred.loc[valid_rows, "pred_q10"].values if valid_rows.any() else None,
        district_pred.loc[valid_rows, "pred_q90"].values if valid_rows.any() else None
    ) if valid_rows.any() else {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan, "Coverage": np.nan, "N": 0}

    prophet_district_rows.append({
        "DistrictId": item_id,
        "DistrictName": district_name,
        "changepoint_prior_scale": cfg["changepoint_prior_scale"],
        "seasonality_prior_scale": cfg["seasonality_prior_scale"],
        "seasonality_mode": cfg["seasonality_mode"],
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "MAPE": m["MAPE"],
        "Coverage": m["Coverage"],
        "N": m["N"]
    })

prophet_predictions_df = pd.concat(all_prophet_predictions, ignore_index=True)
prophet_errors_df = pd.concat(all_prophet_errors, ignore_index=True)
prophet_district_metrics_df = pd.DataFrame(prophet_district_rows).sort_values("DistrictName").reset_index(drop=True)

overall_valid = prophet_errors_df["pred_q50"].notna()
overall_metrics = compute_metrics(
    prophet_errors_df.loc[overall_valid, "target"].values,
    prophet_errors_df.loc[overall_valid, "pred_q50"].values,
    prophet_errors_df.loc[overall_valid, "pred_q10"].values if overall_valid.any() else None,
    prophet_errors_df.loc[overall_valid, "pred_q90"].values if overall_valid.any() else None
) if overall_valid.any() else {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan, "Coverage": np.nan, "N": 0}

overall_row = pd.DataFrame([{
    "DistrictId": "Overall",
    "DistrictName": "Overall",
    "changepoint_prior_scale": "district-specific",
    "seasonality_prior_scale": "district-specific",
    "seasonality_mode": "district-specific",
    "MAE": overall_metrics["MAE"],
    "RMSE": overall_metrics["RMSE"],
    "MAPE": overall_metrics["MAPE"],
    "Coverage": overall_metrics["Coverage"],
    "N": overall_metrics["N"]
}])

final_prophet_test_metrics_df = pd.concat([prophet_district_metrics_df, overall_row], ignore_index=True)

# save outputs
prophet_predictions_df.to_csv(PROPHET_PREDICTIONS_CSV, index=False)
prophet_errors_df.to_csv(PROPHET_STEP_ERRORS_CSV, index=False)
final_prophet_test_metrics_df.to_csv(PROPHET_TEST_METRICS_CSV, index=False)

print("\nProphet predicted rows:", len(prophet_predictions_df))
print("Expected test rows:", len(test_prophet_df_sorted))

print("\nFinal Prophet 2024 test metrics:")
display(final_prophet_test_metrics_df)

print("\nSaved Prophet files:")
print(PROPHET_VAL_RESULTS_CSV)
print(PROPHET_BEST_CONFIGS_CSV)
print(PROPHET_PREDICTIONS_CSV)
print(PROPHET_STEP_ERRORS_CSV)
print(PROPHET_TEST_METRICS_CSV)